In [1]:
import os
import sys

from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import Callback
from pytorch_lightning.callbacks import EarlyStopping
from pytorch_lightning.callbacks import DeviceStatsMonitor, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from models.dvRespNet import EventRespirationNet
from aedat_dataset import AEDATRespirationDataset
from datamodule import AEDATDataModule 
import matplotlib.pyplot as plt


In [2]:
def plot_loss_curves(train_losses, val_losses):
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Loss', marker='o')
    plt.plot(val_losses, label='Validation Loss', marker='x')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig("loss_curve.png")
    plt.show()
    
    
    
def plot_predictions(y_true, y_pred, title="Predicted vs Ground Truth RR"):
    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, c='blue', alpha=0.6, label='Predictions')
    plt.plot([min(y_true), max(y_true)], [min(y_true), max(y_true)], 'r--', label='Ideal')
    plt.xlabel('Ground Truth RR')
    plt.ylabel('Predicted RR')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()
    
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=20,
    mode="min",
    verbose=True
)

class LossHistory(Callback):
    def __init__(self):
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("train_loss")
        if loss is not None:
            self.train_losses.append(loss.item())

    def on_validation_epoch_end(self, trainer, pl_module):
        loss = trainer.callback_metrics.get("val_loss")
        if loss is not None:
            self.val_losses.append(loss.item())


checkpoint_callback = ModelCheckpoint(
    dirpath="cpks",
    filename="best_model-{epoch:02d}-{val_loss:.2f}",
    monitor="val_loss",
    save_top_k=1,
    mode="min"
)


In [3]:
    # 1) Instantiate model
    model = EventRespirationNet(lr=5e-5)
    loss_history_cb = LossHistory()

    # 2) Instantiate DataModule pointing at your AEDAT folders and GT CSV.
    #    data_dirs maps each dataset name (matching the CSV's "Dataset" column)
    #    to its recordings directory. Place the 042026 recordings under
    #    ../data/042026 (same convention as 042025).
    data_module = AEDATDataModule(
        data_dirs={
            "042025": "../data/042025",
            "042026": "../data/042026",
        },
        csv_path="../data/ground_truth.csv",
        batch_size=1, 
        frames_per_sample=100,
        max_time=10,
        filtered=False,
        plot_labels = False
    )

    # 3) Trainer
    trainer = Trainer(
        max_epochs=200,
        callbacks=[loss_history_cb, DeviceStatsMonitor(), checkpoint_callback],
        accelerator="auto",
        devices="auto",
        log_every_n_steps=10
    )

    # 4) Fit
    trainer.fit(model, datamodule=data_module)

    trainer.test(model, datamodule=data_module)
    
    plot_loss_curves(loss_history_cb.train_losses, loss_history_cb.val_losses)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/opt/conda/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A100 80GB PCIe') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | '

[AEDATRespirationDataset] matched 458 files from 2 dataset(s).


/opt/conda/lib/python3.11/site-packages/pytorch_lightning/core/optimizer.py:378: Found unsupported keys in the optimizer configuration: {'gradient_clip_val'}

  | Name      | Type       | Params | Mode  | FLOPs
---------------------------------------------------------
0 | encoder   | Sequential | 357 K  | train | 0    
1 | lstm      | LSTM       | 1.2 M  | train | 0    
2 | regressor | Sequential | 8.3 K  | train | 0    
---------------------------------------------------------
1.5 M     Trainable params
0         Non-trainable params
1.5 M     Total params
6.190     Total estimated model params size (MB)
89        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


OutOfMemoryError: CUDA out of memory. Tried to allocate 854.00 MiB. GPU 0 has a total capacty of 79.25 GiB of which 316.94 MiB is free. Process 3125844 has 1.44 GiB memory in use. Including non-PyTorch memory, this process has 996.00 MiB memory in use. Of the allocated memory 474.67 MiB is allocated by PyTorch, and 19.33 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF